# Two Speeds of Photometry Ingestion

SkyPortal stores both when a photometric measurement occurred and when that row
entered the database. This notebook measures the difference directly from the
frozen raw capture. It tests whether GCN-transcribed measurements and all other
rows exhibit distinct ingestion speeds, then checks censoring and bulk uploads
before drawing a conclusion.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DETAIL_ROOT = ROOT / "data" / "raw" / "skyportal" / "source_detail_20260724"
EVIDENCE_PATH = ROOT / "notebooks" / "evidence" / "04_photometry_lag.csv"
GCN_ARCHIVE_RUNS = sorted(
    (ROOT / "data" / "raw" / "gcn" / "circulars" / "archive_json").glob(
        "*/extracted/archive.json"
    )
)
if not GCN_ARCHIVE_RUNS:
    raise FileNotFoundError("No extracted raw GCN circular archive is available")
GCN_ARCHIVE = GCN_ARCHIVE_RUNS[-1]
CAPTURE_DATE = pd.to_datetime(
    DETAIL_ROOT.name.rsplit("_", maxsplit=1)[-1], format="%Y%m%d", utc=True
)

raw_entries = []
photometry_files = sorted(DETAIL_ROOT.glob("*/photometry.json"))
for photometry_path in photometry_files:
    envelope = json.loads(photometry_path.read_text(encoding="utf-8"))
    payload_data = envelope["payload"]["data"]
    if isinstance(payload_data, dict):
        payload_data = payload_data.get(
            "photometry", payload_data.get("data", [])
        )
    for raw_record in payload_data:
        raw_entries.append(
            {
                "source_id": photometry_path.parent.name,
                "raw_record": raw_record,
                **raw_record,
            }
        )

photometry = pd.DataFrame(raw_entries)
print(f"Photometry files: {len(photometry_files)}")
print(f"Raw photometry rows: {len(photometry)}")
print(f"Capture date: {CAPTURE_DATE.date().isoformat()}")
print(f"Raw GCN archive: {GCN_ARCHIVE.relative_to(ROOT)}")

Photometry files: 799
Raw photometry rows: 7968
Capture date: 2026-07-24
Raw GCN archive: data/raw/gcn/circulars/archive_json/20260720_093324/extracted/archive.json


## 1. The question

**QUESTION.** How quickly does a physical photometric measurement become
available in SkyPortal, and does that delay differ between rows explicitly
marked as GCN transcriptions and all other origins?

In [2]:
measurement_definition = pd.DataFrame(
    [
        ("occurrence time", "mjd", "UTC instant of the observation"),
        ("knowledge time", "created_at", "UTC instant when SkyPortal stored the row"),
        ("lag", "created_at - mjd", "Hours between occurrence and availability"),
        ("GCN-origin", "origin.strip() == 'GCN'", "Explicit GCN provenance only"),
        ("other-origin", "all remaining origin values", "No inferred GCN membership"),
    ],
    columns=["concept", "raw definition", "interpretation"],
)
print(measurement_definition.to_string(index=False))

        concept              raw definition                            interpretation
occurrence time                         mjd            UTC instant of the observation
 knowledge time                  created_at UTC instant when SkyPortal stored the row
            lag            created_at - mjd Hours between occurrence and availability
     GCN-origin     origin.strip() == 'GCN'              Explicit GCN provenance only
   other-origin all remaining origin values                No inferred GCN membership


**FINDING.** The raw schema supports a direct lag measurement. The grouping is
intentionally conservative: only exact, whitespace-normalized `GCN` values are
called GCN-origin. Origin strings that merely contain a circular number remain
in the other-origin group.

## 2. Anatomy of a raw photometry row

**QUESTION.** Which raw fields establish observation time, knowledge time,
provenance, and instrument, and does `altdata` contain direct circular
references?

In [3]:
def altdata_text(value):
    return json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else ""

complete_example = next(
    entry
    for entry in raw_entries
    if str(entry["origin"]).strip() == "GCN"
    and "gcn" in altdata_text(entry["altdata"]).lower()
)
print("Complete raw row:")
print(json.dumps(complete_example["raw_record"], indent=2, ensure_ascii=False))

origin_counts = (
    photometry["origin"]
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("origin")
    .reset_index(name="rows")
)
origin_counts = origin_counts.sort_values(
    ["rows", "origin"], ascending=[False, True], kind="mergesort"
)
print("\nAll raw origin values:")
print(origin_counts.to_string(index=False))

field_roles = []
for role, field_name in [
    ("observation_time", "mjd"),
    ("knowledge_time", "created_at"),
    ("provenance", "origin"),
    ("instrument_identifier", "instrument_id"),
    ("instrument_name", "instrument_name"),
]:
    field_roles.append(
        {
            "role": role,
            "field": field_name,
            "pandas_dtype": str(photometry[field_name].dtype),
            "python_types": ", ".join(
                sorted({type(value).__name__ for value in photometry[field_name]})
            ),
            "populated": int(photometry[field_name].notna().sum()),
        }
    )
print("\nField roles and observed types:")
print(pd.DataFrame(field_roles).to_string(index=False))

altdata_with_gcn = photometry[
    photometry["altdata"].map(
        lambda value: "gcn" in altdata_text(value).lower()
    )
]
explicit_url = altdata_with_gcn[
    altdata_with_gcn["altdata"].map(
        lambda value: "gcn.nasa.gov/circulars/" in altdata_text(value).lower()
    )
].iloc[0]
stdview_example = photometry[
    photometry["altdata"].map(
        lambda value: isinstance(value, dict) and "stdview" in value
    )
].iloc[0]
altdata_examples = [
    altdata_with_gcn.iloc[0],
    explicit_url,
    stdview_example,
]
print(f"\nRows with a GCN reference in altdata: {len(altdata_with_gcn)}")
for example_number, row in enumerate(altdata_examples, start=1):
    print(
        f"altdata example {example_number}: source_id={row['source_id']} "
        f"row_id={row['id']} origin={row['origin']}"
    )
    print(json.dumps(row["altdata"], indent=2, ensure_ascii=False))

created_at_probe = pd.to_datetime(photometry["created_at"], utc=True, errors="raise")
print(f"\nMinimum created_at: {created_at_probe.min().isoformat()}")
print(f"Maximum created_at: {created_at_probe.max().isoformat()}")
print(f"Capture date from directory: {CAPTURE_DATE.date().isoformat()}")

Complete raw row:
{
  "obj_id": "2025gcz",
  "ra": null,
  "dec": null,
  "filter": "sdssi",
  "mjd": 60761.88494212963,
  "snr": 271.43405118953234,
  "instrument_id": 91,
  "instrument_name": "GCN",
  "ra_unc": null,
  "dec_unc": null,
  "origin": "GCN",
  "id": 60104,
  "altdata": {
    "note": "GCN 39894 1.6m Mephisto optical observations, arbitrary upper limit",
    "exposure": "2x80"
  },
  "created_at": "2025-03-28T07:02:40.665323",
  "groups": [
    {
      "id": 3,
      "name": "GRANDMA",
      "nickname": null,
      "single_user_group": false
    },
    {
      "id": 36,
      "name": "candrade",
      "nickname": null,
      "single_user_group": true
    }
  ],
  "mag": 14.73,
  "magerr": 0.004,
  "magsys": "ab",
  "limiting_mag": 15.0
}

All raw origin values:
                                              origin  rows
                                                  fp  2786
                                                None  2135
                                      

**FINDING.** All 7,968 rows carry `mjd`, `created_at`, `origin`,
`instrument_id`, and `instrument_name`. The capture has 75 distinct raw origin
strings. `altdata` mentions a GCN reference in 1,239 rows, including explicit
circular URLs, so publication-to-entry lag can be measured for unambiguous
references rather than approximated.

## 3. Lag distribution by provenance

**QUESTION.** What is the distribution of `created_at - observation_time` for
GCN-origin and other-origin rows, and are impossible negative lags present?

In [4]:
MJD_EPOCH = pd.Timestamp("1858-11-17", tz="UTC")
photometry["observation_time_utc"] = MJD_EPOCH + pd.to_timedelta(
    photometry["mjd"], unit="D"
)
photometry["created_at_utc"] = pd.to_datetime(
    photometry["created_at"], utc=True, errors="raise"
)
photometry["lag_hours"] = (
    photometry["created_at_utc"] - photometry["observation_time_utc"]
).dt.total_seconds() / 3600.0
photometry["provenance"] = np.where(
    photometry["origin"].astype(str).str.strip().str.casefold().eq("gcn"),
    "GCN-origin",
    "other-origin",
)

control = photometry[
    (photometry["source_id"] == "GRB241030") & (photometry["id"] == 37670)
].iloc[0]
print("MJD conversion control:")
print(
    f"source_id={control['source_id']} row_id={control['id']} "
    f"mjd={control['mjd']} "
    f"observation_time_utc={control['observation_time_utc'].isoformat()} "
    f"created_at={control['created_at']} lag_hours={control['lag_hours']:.3f}"
)
print("Expected: mjd=60613.2435 created_at=2024-10-30T16:56:27.973714 lag_hours=11.097")

summary_rows = []
for provenance, group in photometry.groupby("provenance", sort=True):
    quantiles = group["lag_hours"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    summary_rows.append(
        {
            "provenance": provenance,
            "n": len(group),
            "min_hours": group["lag_hours"].min(),
            "p10_hours": quantiles.loc[0.10],
            "p25_hours": quantiles.loc[0.25],
            "median_hours": quantiles.loc[0.50],
            "median_days": quantiles.loc[0.50] / 24.0,
            "p75_hours": quantiles.loc[0.75],
            "p90_hours": quantiles.loc[0.90],
            "max_hours": group["lag_hours"].max(),
        }
    )
lag_summary = pd.DataFrame(summary_rows)
print("\nLag distribution by provenance:")
print(lag_summary.round(3).to_string(index=False))

negative_lags = photometry[photometry["lag_hours"] < 0].copy()
negative_pct = 100.0 * len(negative_lags) / len(photometry)
print(
    f"\nNegative lags: {len(negative_lags)}/{len(photometry)} "
    f"({negative_pct:.3f}%)"
)
if not negative_lags.empty:
    print("Three most negative full raw rows:")
    for _, row in negative_lags.sort_values(
        ["lag_hours", "source_id", "id"], kind="mergesort"
    ).head(3).iterrows():
        print(
            f"source_id={row['source_id']} row_id={row['id']} "
            f"lag_hours={row['lag_hours']:.3f}"
        )
        print(json.dumps(row["raw_record"], indent=2, ensure_ascii=False))

quantile_targets = [
    ("p10", 0.10),
    ("p25", 0.25),
    ("median", 0.50),
    ("p75", 0.75),
    ("p90", 0.90),
]
sample_rows = []
for provenance, group in photometry.groupby("provenance", sort=True):
    for quantile_name, quantile_value in quantile_targets:
        target_lag = group["lag_hours"].quantile(quantile_value)
        selected = (
            group.assign(distance=(group["lag_hours"] - target_lag).abs())
            .sort_values(
                ["distance", "lag_hours", "source_id", "id"],
                kind="mergesort",
            )
            .iloc[0]
        )
        sample_rows.append(
            {
                "provenance": provenance,
                "quantile": quantile_name,
                "source_id": selected["source_id"],
                "mjd": selected["mjd"],
                "observation_time_utc": selected["observation_time_utc"].isoformat(),
                "created_at": selected["created_at"],
                "lag_hours": selected["lag_hours"],
                "instrument": selected["instrument_name"],
            }
        )
quantile_samples = pd.DataFrame(sample_rows)
print("\nDeterministic rows nearest fixed quantiles:")
print(quantile_samples.round({"mjd": 6, "lag_hours": 3}).to_string(index=False))

evidence = pd.DataFrame(
    {
        "source_id": photometry["source_id"],
        "provenance": photometry["provenance"],
        "mjd": photometry["mjd"],
        "observation_time_utc": photometry["observation_time_utc"].map(
            lambda value: value.isoformat()
        ),
        "created_at": photometry["created_at_utc"].map(
            lambda value: value.isoformat()
        ),
        "lag_hours": photometry["lag_hours"],
        "instrument": photometry["instrument_name"],
        "band_raw": photometry["filter"],
    }
)
evidence.to_csv(EVIDENCE_PATH, index=False)
print(f"\nEvidence shape: {evidence.shape}")
print(f"Evidence header: {list(evidence.columns)}")
print(f"Evidence path: {EVIDENCE_PATH.relative_to(ROOT)}")

MJD conversion control:
source_id=GRB241030 row_id=37670 mjd=60613.2435 observation_time_utc=2024-10-30T05:50:38.399999732+00:00 created_at=2024-10-30T16:56:27.973714 lag_hours=11.097
Expected: mjd=60613.2435 created_at=2024-10-30T16:56:27.973714 lag_hours=11.097

Lag distribution by provenance:
  provenance    n  min_hours  p10_hours  p25_hours  median_hours  median_days  p75_hours  p90_hours   max_hours
  GCN-origin  997      -2.75      6.533     15.899       104.418        4.351   1203.271   5741.880   16261.271
other-origin 6971  -34855.87     23.508    120.293      1105.197       46.050   2981.270   6326.995 1405188.231

Negative lags: 16/7968 (0.201%)
Three most negative full raw rows:
source_id=ZTF23aaptsuy row_id=3000 lag_hours=-34855.870
{
  "obj_id": "ZTF23aaptsuy",
  "ra": null,
  "dec": null,
  "filter": "ztfr",
  "mjd": 61582.21003,
  "snr": -0.5714401077674365,
  "instrument_id": 1,
  "instrument_name": "ZTF",
  "ra_unc": null,
  "dec_unc": null,
  "origin": "None",
  "id


Deterministic rows nearest fixed quantiles:
  provenance quantile         source_id          mjd                observation_time_utc                 created_at  lag_hours  instrument
  GCN-origin      p10 GRB-250617_210150 60844.100394 2025-06-18T02:24:33.999999652+00:00 2025-06-18T08:58:50.401402      6.571         GCN
  GCN-origin      p25 GCN-250727_105339 60883.531528 2025-07-27T12:45:24.000191688+00:00 2025-07-28T04:39:22.059285     15.899         GCN
  GCN-origin   median GCN-251013_173943 60961.836134 2025-10-13T20:04:01.999999571+00:00 2025-10-18T04:29:08.360767    104.418         GCN
  GCN-origin      p75 GRB-251126_191037 61006.147222 2025-11-27T03:31:59.999999972+00:00 2026-01-16T06:48:17.128392   1203.271 COLIBRI-VIS
  GCN-origin      p90         GRB241026 60611.972222 2024-10-28T23:19:59.999807982+00:00 2025-06-24T17:14:06.699854   5729.902         GCN
other-origin      p10      ZTF24abisvfd 60576.482025 2024-09-23T11:34:07.003193200+00:00 2024-09-24T11:04:34.486387     2

**FINDING.** The unadjusted medians differ by a factor of 10.58:
GCN-origin is 104.418 hours (4.351 days; n=997), while other-origin is
1,105.197 hours (46.050 days; n=6,971). This supports an aggregate gap, but
GCN-origin is measured in days rather than uniformly in hours. Sixteen rows
(0.201%) have impossible negative lags and are retained as visible data-quality
failures rather than silently removed.

## 4. Controlling the two confounders

The raw difference can be distorted by right censoring near the capture date or
by retrospective uploads concentrated on a few database dates.

### 4.1 Censoring

**QUESTION.** Does the fixed capture date prevent recent rows from exhibiting
long lags, thereby making one provenance group appear artificially faster?

In [5]:
photometry["max_observable_lag_days"] = (
    CAPTURE_DATE - photometry["observation_time_utc"]
).dt.total_seconds() / 86400.0

censoring_rows = []
for threshold_days in [30, 90, 180]:
    censored = photometry["max_observable_lag_days"] < threshold_days
    censoring_rows.append(
        {
            "threshold_days": threshold_days,
            "censored_rows": int(censored.sum()),
            "censored_pct": 100.0 * censored.mean(),
        }
    )
censoring = pd.DataFrame(censoring_rows)
print("Censoring at the fixed capture date:")
print(censoring.round(3).to_string(index=False))

restricted_180 = photometry[
    photometry["max_observable_lag_days"] >= 180
].copy()
comparison_rows = []
for provenance in sorted(photometry["provenance"].unique()):
    all_group = photometry[photometry["provenance"] == provenance]
    restricted_group = restricted_180[
        restricted_180["provenance"] == provenance
    ]
    comparison_rows.append(
        {
            "provenance": provenance,
            "all_n": len(all_group),
            "all_median_hours": all_group["lag_hours"].median(),
            "all_median_days": all_group["lag_hours"].median() / 24.0,
            "eligible_180d_n": len(restricted_group),
            "eligible_180d_median_hours": restricted_group["lag_hours"].median(),
            "eligible_180d_median_days": restricted_group["lag_hours"].median() / 24.0,
        }
    )
censoring_comparison = pd.DataFrame(comparison_rows)
print("\nUnrestricted versus at least 180 observable days:")
print(censoring_comparison.round(3).to_string(index=False))

Censoring at the fixed capture date:
 threshold_days  censored_rows  censored_pct
             30            135         1.694
             90            597         7.492
            180            864        10.843

Unrestricted versus at least 180 observable days:
  provenance  all_n  all_median_hours  all_median_days  eligible_180d_n  eligible_180d_median_hours  eligible_180d_median_days
  GCN-origin    997           104.418            4.351              587                      78.011                      3.250
other-origin   6971          1105.197           46.050             6517                    1254.865                     52.286


**FINDING.** Only 864 rows (10.843%) lack 180 days of observable
lag. Restricting to older observations changes the GCN-origin median from 4.351
to 3.250 days and the other-origin median from 46.050 to 52.286 days. The gap
therefore does not result from recent-row censoring.

### 4.2 Bulk uploads

**QUESTION.** Do long lags represent a steady slower channel, or are they
concentrated in a small number of retrospective ingestion days?

In [6]:
photometry["created_day"] = photometry["created_at_utc"].dt.strftime(
    "%Y-%m-%d"
)
created_day_distribution = (
    photometry["created_day"]
    .value_counts()
    .rename_axis("created_day")
    .reset_index(name="rows")
)
created_day_distribution = created_day_distribution.sort_values(
    ["rows", "created_day"], ascending=[False, True], kind="mergesort"
)
top_upload_days = created_day_distribution.head(10).copy()
top_upload_share = 100.0 * top_upload_days["rows"].sum() / len(photometry)
print("Ten largest created_at days:")
print(top_upload_days.to_string(index=False))
print(
    f"Top ten days: {int(top_upload_days['rows'].sum())}/{len(photometry)} "
    f"rows ({top_upload_share:.3f}%)"
)

without_top_days = photometry[
    ~photometry["created_day"].isin(top_upload_days["created_day"])
].copy()
bulk_comparison_rows = []
for provenance in sorted(photometry["provenance"].unique()):
    included = photometry[photometry["provenance"] == provenance]
    excluded = without_top_days[without_top_days["provenance"] == provenance]
    bulk_comparison_rows.append(
        {
            "provenance": provenance,
            "including_n": len(included),
            "including_median_hours": included["lag_hours"].median(),
            "including_median_days": included["lag_hours"].median() / 24.0,
            "excluding_top_days_n": len(excluded),
            "excluding_top_days_median_hours": excluded["lag_hours"].median(),
            "excluding_top_days_median_days": excluded["lag_hours"].median() / 24.0,
        }
    )
bulk_comparison = pd.DataFrame(bulk_comparison_rows)
print("\nMedians including and excluding the ten largest upload days:")
print(bulk_comparison.round(3).to_string(index=False))

print("\nProvenance composition of the ten largest upload days:")
print(
    pd.crosstab(
        photometry.loc[
            photometry["created_day"].isin(top_upload_days["created_day"]),
            "created_day",
        ],
        photometry.loc[
            photometry["created_day"].isin(top_upload_days["created_day"]),
            "provenance",
        ],
    ).to_string()
)

Ten largest created_at days:
created_day  rows
 2024-04-26   742
 2024-07-16   740
 2024-06-27   642
 2022-11-10   469
 2024-01-17   305
 2026-02-14   226
 2024-04-30   157
 2025-09-27   135
 2026-07-03   124
 2026-07-23   124
Top ten days: 3664/7968 rows (45.984%)



Medians including and excluding the ten largest upload days:
  provenance  including_n  including_median_hours  including_median_days  excluding_top_days_n  excluding_top_days_median_hours  excluding_top_days_median_days
  GCN-origin          997                 104.418                  4.351                   925                           90.200                           3.758
other-origin         6971                1105.197                 46.050                  3379                          139.414                           5.809

Provenance composition of the ten largest upload days:
provenance   GCN-origin  other-origin
created_day                          
2022-11-10            0           469
2024-01-17            0           305
2024-04-26            0           742
2024-04-30            0           157
2024-06-27            0           642
2024-07-16            0           740
2025-09-27            0           135
2026-02-14            0           226
2026-07-03           7

**FINDING.** The ten largest upload days contain 3,664 rows
(45.984% of all photometry) and are almost entirely other-origin. Removing those
days moves the other-origin median from 46.050 to 5.809 days, while GCN-origin
moves from 4.351 to 3.758 days. The median ratio falls from 10.58 to 1.55, so a
few mass-ingestion events explain most of the apparent second speed; the data do
not support treating the unadjusted gap as a stable channel property.

### 4.3 Direct GCN publication cross-check

**QUESTION.** For rows whose `altdata` names exactly one circular, how long
after circular publication did the row enter SkyPortal?

In [7]:
CIRCULAR_REFERENCE_PATTERNS = [
    re.compile(r"https?://gcn\.nasa\.gov/circulars/(\d+)", re.IGNORECASE),
    re.compile(
        r"\bGCN(?:\s+CIRCULAR)?\s*[#:]?\s*(\d{4,6})\b",
        re.IGNORECASE,
    ),
]


def circular_references(altdata):
    if not isinstance(altdata, dict):
        return []
    text = json.dumps(altdata, ensure_ascii=False)
    references = []
    for pattern in CIRCULAR_REFERENCE_PATTERNS:
        references.extend(int(value) for value in pattern.findall(text))
    return sorted(set(references))


gcn_rows = photometry[photometry["provenance"] == "GCN-origin"].copy()
gcn_rows["circular_references"] = gcn_rows["altdata"].map(circular_references)
gcn_rows["n_circular_references"] = gcn_rows["circular_references"].map(len)
print("Circular-reference count among GCN-origin rows:")
print(
    gcn_rows["n_circular_references"]
    .value_counts()
    .sort_index()
    .rename_axis("n_references")
    .reset_index(name="rows")
    .to_string(index=False)
)

publication_matches = []
missing_circular_files = []
for row in gcn_rows[gcn_rows["n_circular_references"] == 1].itertuples():
    circular_id = row.circular_references[0]
    circular_path = GCN_ARCHIVE / f"{circular_id}.json"
    if not circular_path.exists():
        missing_circular_files.append(circular_id)
        continue
    circular = json.loads(circular_path.read_text(encoding="utf-8"))
    publication_time = pd.to_datetime(circular["createdOn"], unit="ms", utc=True)
    publication_matches.append(
        {
            "source_id": row.source_id,
            "photometry_id": row.id,
            "circular_id": circular_id,
            "circular_published_at": publication_time,
            "skyportal_created_at": row.created_at_utc,
            "publication_to_entry_hours": (
                row.created_at_utc - publication_time
            ).total_seconds()
            / 3600.0,
        }
    )
publication_lag = pd.DataFrame(publication_matches)
publication_summary = pd.DataFrame(
    [
        {
            "gcn_origin_rows": len(gcn_rows),
            "matched_rows": len(publication_lag),
            "missing_circular_files": len(missing_circular_files),
            "median_hours": publication_lag["publication_to_entry_hours"].median(),
            "median_days": publication_lag["publication_to_entry_hours"].median() / 24.0,
            "p90_hours": publication_lag["publication_to_entry_hours"].quantile(0.90),
            "p90_days": publication_lag["publication_to_entry_hours"].quantile(0.90) / 24.0,
            "negative_rows": int(
                (publication_lag["publication_to_entry_hours"] < 0).sum()
            ),
        }
    ]
)
print("\nDirect circular publication-to-entry comparison:")
print(publication_summary.round(3).to_string(index=False))
print(f"Missing circular ids: {sorted(set(missing_circular_files))}")

Circular-reference count among GCN-origin rows:
 n_references  rows
            0    35
            1   962



Direct circular publication-to-entry comparison:
 gcn_origin_rows  matched_rows  missing_circular_files  median_hours  median_days  p90_hours  p90_days  negative_rows
             997           958                       4        76.695        3.196   5448.715    227.03              0
Missing circular ids: [45189, 410024]


**FINDING.** Of 997 GCN-origin rows, 958 have one unambiguous
reference and a corresponding raw circular. Their median delay from circular
publication to SkyPortal entry is 76.695 hours (3.196 days), and p90 is
5,448.715 hours (227.030 days), with zero negative delays. Some rows enter
within minutes, but the corpus-wide direct comparison contradicts a general
“hours-only” transcription claim.

## 5. Decomposing the GCN channel

**QUESTION.** For directly matched GCN rows, how much of the total
observation-to-SkyPortal lag occurs before circular publication, and how much
occurs while SkyPortal transcribes the published measurement?

In [8]:
matched_photometry = photometry[
    [
        "source_id",
        "id",
        "mjd",
        "observation_time_utc",
        "created_at_utc",
        "lag_hours",
        "raw_record",
    ]
].rename(columns={"id": "photometry_id"})

decomposed_gcn = publication_lag.merge(
    matched_photometry,
    on=["source_id", "photometry_id"],
    how="inner",
    validate="one_to_one",
)
decomposed_gcn["observation_to_publication_hours"] = (
    decomposed_gcn["circular_published_at"]
    - decomposed_gcn["observation_time_utc"]
).dt.total_seconds() / 3600.0
decomposed_gcn["publication_to_skyportal_hours"] = decomposed_gcn[
    "publication_to_entry_hours"
]
decomposed_gcn["observation_to_skyportal_hours"] = decomposed_gcn[
    "lag_hours"
]
decomposed_gcn["component_residual_hours"] = (
    decomposed_gcn["observation_to_publication_hours"]
    + decomposed_gcn["publication_to_skyportal_hours"]
    - decomposed_gcn["observation_to_skyportal_hours"]
)

lag_components = [
    ("A: observation -> circular publication", "observation_to_publication_hours"),
    ("B: circular publication -> SkyPortal", "publication_to_skyportal_hours"),
    ("C: observation -> SkyPortal", "observation_to_skyportal_hours"),
]
decomposition_rows = []
for lag_name, column in lag_components:
    values_hours = decomposed_gcn[column]
    quantiles_hours = values_hours.quantile([0.10, 0.25, 0.50, 0.75, 0.90])
    for unit, divisor in [("hours", 1.0), ("days", 24.0)]:
        decomposition_rows.append(
            {
                "lag": lag_name,
                "unit": unit,
                "n": len(values_hours),
                "min": values_hours.min() / divisor,
                "p10": quantiles_hours.loc[0.10] / divisor,
                "p25": quantiles_hours.loc[0.25] / divisor,
                "median": quantiles_hours.loc[0.50] / divisor,
                "p75": quantiles_hours.loc[0.75] / divisor,
                "p90": quantiles_hours.loc[0.90] / divisor,
                "max": values_hours.max() / divisor,
            }
        )
decomposition_summary = pd.DataFrame(decomposition_rows)
print("GCN channel decomposition:")
print(decomposition_summary.round(3).to_string(index=False))

residual_tolerance_hours = 1e-9
component_failures = decomposed_gcn[
    decomposed_gcn["component_residual_hours"].abs()
    > residual_tolerance_hours
]
print()
print(
    f"A + B = C failures above {residual_tolerance_hours} h: "
    f"{len(component_failures)}/{len(decomposed_gcn)}"
)
print(
    f"Maximum absolute component residual: "
    f"{decomposed_gcn['component_residual_hours'].abs().max():.12g} h"
)


def decomposed_record(row):
    return {
        "source_id": row["source_id"],
        "photometry_id": int(row["photometry_id"]),
        "mjd": float(row["mjd"]),
        "observation_time_utc": row["observation_time_utc"].isoformat(),
        "circular_id": int(row["circular_id"]),
        "circular_published_at": row["circular_published_at"].isoformat(),
        "created_at": row["created_at_utc"].isoformat(),
        "lag_A_hours": float(row["observation_to_publication_hours"]),
        "lag_B_hours": float(row["publication_to_skyportal_hours"]),
        "lag_C_hours": float(row["observation_to_skyportal_hours"]),
        "component_residual_hours": float(row["component_residual_hours"]),
        "raw_photometry": row["raw_record"],
    }


if not component_failures.empty:
    print("Three component failures:")
    for _, row in component_failures.head(3).iterrows():
        print(json.dumps(decomposed_record(row), indent=2, ensure_ascii=False))

negative_publication_lags = decomposed_gcn[
    decomposed_gcn["observation_to_publication_hours"] < 0
].copy()
negative_publication_pct = (
    100.0 * len(negative_publication_lags) / len(decomposed_gcn)
)
print()
print(
    f"Negative lag A: {len(negative_publication_lags)}/{len(decomposed_gcn)} "
    f"({negative_publication_pct:.3f}%)"
)
if not negative_publication_lags.empty:
    print("Three most negative lag-A rows:")
    for _, row in negative_publication_lags.sort_values(
        ["observation_to_publication_hours", "source_id", "photometry_id"],
        kind="mergesort",
    ).head(3).iterrows():
        print(json.dumps(decomposed_record(row), indent=2, ensure_ascii=False))

for quantile_name, quantile_value in [("median", 0.50), ("p90", 0.90)]:
    target = decomposed_gcn["observation_to_publication_hours"].quantile(
        quantile_value
    )
    nearest_rows = (
        decomposed_gcn.assign(
            distance=(
                decomposed_gcn["observation_to_publication_hours"] - target
            ).abs()
        )
        .sort_values(
            [
                "distance",
                "observation_to_publication_hours",
                "source_id",
                "photometry_id",
            ],
            kind="mergesort",
        )
        .head(5)
    )
    print()
    print(f"Five rows nearest lag-A {quantile_name} ({target:.6f} h):")
    for _, row in nearest_rows.iterrows():
        print(json.dumps(decomposed_record(row), indent=2, ensure_ascii=False))

GCN channel decomposition:
                                   lag  unit   n         min   p10    p25  median      p75      p90        max
A: observation -> circular publication hours 958 -176382.692 1.141  2.672  10.732   22.832   58.269    814.153
A: observation -> circular publication  days 958   -7349.279 0.048  0.111   0.447    0.951    2.428     33.923
  B: circular publication -> SkyPortal hours 958       0.068 0.697  2.578  76.695 1212.127 5448.715 176389.168
  B: circular publication -> SkyPortal  days 958       0.003 0.029  0.107   3.196   50.505  227.030   7349.549
           C: observation -> SkyPortal hours 958      -2.750 6.339 15.142 100.982 1226.389 5648.918  16261.271
           C: observation -> SkyPortal  days 958      -0.115 0.264  0.631   4.208   51.100  235.372    677.553

A + B = C failures above 1e-09 h: 0/958
Maximum absolute component residual: 2.21049845095e-11 h

Negative lag A: 10/958 (1.044%)
Three most negative lag-A rows:
{
  "source_id": "GRB-260101_0056

**FINDING.** Across 958 directly matched rows, the median GCN-channel
latency A is 10.732 hours (0.447 days), with p90 at 58.269 hours (2.428
days). SkyPortal transcription latency B has a median of 76.695 hours (3.196
days), while total latency C has a 100.982-hour (4.208-day) median. Every row
satisfies A + B = C within `1e-9` hour. Ten lag-A values (1.044%) are negative,
including extreme mismatches between recent photometry and much older circular
ids; they remain visible as reference/data-quality failures.

## 6. What are the bulk days?

**QUESTION.** Are the ten largest SkyPortal creation dates one migration cluster,
or repeated batch-ingestion episodes distributed across the capture history?

In [9]:
bulk_profile_rows = []
for created_day in top_upload_days["created_day"]:
    day_rows = photometry[photometry["created_day"] == created_day].copy()
    observation_ages_days = day_rows["lag_hours"] / 24.0
    origin_breakdown = {
        str(origin): int(count)
        for origin, count in day_rows["origin"].value_counts().sort_index().items()
    }
    bulk_profile_rows.append(
        {
            "created_day": created_day,
            "rows": len(day_rows),
            "distinct_source_ids": day_rows["source_id"].nunique(),
            "distinct_instruments": day_rows["instrument_name"].nunique(),
            "origin_breakdown": json.dumps(
                origin_breakdown, ensure_ascii=False, sort_keys=True
            ),
            "observation_date_min": day_rows[
                "observation_time_utc"
            ].min().date().isoformat(),
            "observation_date_median": day_rows[
                "observation_time_utc"
            ].median().date().isoformat(),
            "observation_date_max": day_rows[
                "observation_time_utc"
            ].max().date().isoformat(),
            "observation_age_min_days": observation_ages_days.min(),
            "observation_age_median_days": observation_ages_days.median(),
            "observation_age_max_days": observation_ages_days.max(),
        }
    )
bulk_day_profile = pd.DataFrame(bulk_profile_rows)
print("Bulk-day profiles:")
print(
    bulk_day_profile.round(
        {
            "observation_age_min_days": 3,
            "observation_age_median_days": 3,
            "observation_age_max_days": 3,
        }
    ).to_string(index=False)
)

chronological_bulk_days = sorted(
    pd.to_datetime(top_upload_days["created_day"], utc=True)
)
bulk_calendar_span_days = (
    chronological_bulk_days[-1] - chronological_bulk_days[0]
).days
bulk_pairwise_gaps_days = [
    (later - earlier).days
    for earlier, later in zip(
        chronological_bulk_days, chronological_bulk_days[1:]
    )
]
print()
print(
    f"Calendar span: {chronological_bulk_days[0].date().isoformat()} to "
    f"{chronological_bulk_days[-1].date().isoformat()} "
    f"({bulk_calendar_span_days} days)"
)
print(f"Chronological pairwise gaps in days: {bulk_pairwise_gaps_days}")

largest_bulk_day = top_upload_days.iloc[0]["created_day"]
largest_day_rows = photometry[
    photometry["created_day"] == largest_bulk_day
]
largest_day_top_sources = (
    largest_day_rows["source_id"]
    .value_counts()
    .head(5)
    .rename_axis("source_id")
    .reset_index(name="rows")
)
print()
print(f"Five largest source contributions on {largest_bulk_day}:")
print(largest_day_top_sources.to_string(index=False))

Bulk-day profiles:
created_day  rows  distinct_source_ids  distinct_instruments                                    origin_breakdown observation_date_min observation_date_median observation_date_max  observation_age_min_days  observation_age_median_days  observation_age_max_days
 2024-04-26   742                    1                     2                           {"fp": 741, "stdview": 1}           2023-03-28              2023-12-27           2024-04-26                     0.143                      121.383                   395.550
 2024-07-16   740                    2                     1                                         {"fp": 740}           2024-01-25              2024-04-17           2024-07-15                     1.506                       89.560                   173.157
 2024-06-27   642                    4                     3                {"GRANDMA": 1, "None": 1, "fp": 640}           2023-05-28              2024-03-20           2024-06-27                     0.

**FINDING.** The ten dates span 1,351 calendar days, with chronological
gaps of 433, 100, 4, 58, 19, 438, 140, 139, and 20 days. They are repeated
batch-ingestion dates spread over several years, not one contiguous migration
cluster. Five dates contain only one source, and the largest date consists of
742 rows from `EP240426a`; several batches load observations tens to hundreds
of days old. This describes a recurring, source-concentrated ingestion pattern
without assigning its operational cause.

## 7. Provenance reconciliation

**QUESTION.** How do the lag comparisons change when GCN provenance is recognized
from either the raw `origin` value or a GCN reference inside `altdata`?

In [10]:
photometry["has_gcn_altdata_reference"] = photometry["altdata"].map(
    lambda value: isinstance(value, dict)
    and "gcn" in json.dumps(value, ensure_ascii=False).lower()
)
photometry["corrected_provenance"] = np.where(
    photometry["provenance"].eq("GCN-origin")
    | photometry["has_gcn_altdata_reference"],
    "GCN-signaled",
    "other-origin",
)
corrected_group_sizes = (
    photometry["corrected_provenance"]
    .value_counts()
    .rename_axis("corrected_provenance")
    .reset_index(name="rows")
)
print("Corrected provenance group sizes:")
print(corrected_group_sizes.to_string(index=False))

control_masks = [
    ("unadjusted", pd.Series(True, index=photometry.index)),
    (">=180 observable days", photometry["max_observable_lag_days"] >= 180),
    (
        "bulk days excluded",
        ~photometry["created_day"].isin(top_upload_days["created_day"]),
    ),
]
corrected_comparison_rows = []
for control_name, mask in control_masks:
    selected = photometry[mask]
    old_medians = (
        selected.groupby("provenance", sort=True)["lag_hours"].median() / 24.0
    )
    new_medians = (
        selected.groupby("corrected_provenance", sort=True)["lag_hours"].median()
        / 24.0
    )
    corrected_comparison_rows.append(
        {
            "control": control_name,
            "n": len(selected),
            "old_gcn_days": old_medians["GCN-origin"],
            "old_other_days": old_medians["other-origin"],
            "old_ratio": (
                old_medians["other-origin"] / old_medians["GCN-origin"]
            ),
            "new_gcn_days": new_medians["GCN-signaled"],
            "new_other_days": new_medians["other-origin"],
            "new_ratio": (
                new_medians["other-origin"] / new_medians["GCN-signaled"]
            ),
        }
    )
corrected_comparison = pd.DataFrame(corrected_comparison_rows)
print()
print("Old and reconciled provenance comparisons:")
print(corrected_comparison.round(3).to_string(index=False))

Corrected provenance group sizes:
corrected_provenance  rows
        other-origin  6699
        GCN-signaled  1269

Old and reconciled provenance comparisons:
              control    n  old_gcn_days  old_other_days  old_ratio  new_gcn_days  new_other_days  new_ratio
           unadjusted 7968         4.351          46.050     10.584         4.486          45.305     10.098
>=180 observable days 7104         3.250          52.286     16.086         4.226          52.178     12.346
   bulk days excluded 4304         3.758           5.809      1.546         3.882           5.808      1.496


**FINDING.** The union produces 1,269 GCN-signaled and 6,699 other
rows. Reclassification changes the unadjusted ratio from 10.584 to 10.098, the
180-day ratio from 16.086 to 12.346, and the bulk-excluded ratio from 1.546 to
1.496. It does not materially change the observation-to-SkyPortal comparison;
the decisive change comes from measuring first GCN availability at circular
publication rather than at later SkyPortal transcription.

## 8. Decision

**QUESTION.** When each channel is timed at its own point of availability, does
the frozen capture support a two-speeds finding, and what role do bulk ingestion
dates play?

In [11]:
gcn_channel_median_hours = decomposed_gcn[
    "observation_to_publication_hours"
].median()
gcn_channel_p90_hours = decomposed_gcn[
    "observation_to_publication_hours"
].quantile(0.90)
skyportal_transcription_median_days = decomposed_gcn[
    "publication_to_skyportal_hours"
].median() / 24.0

unadjusted_corrected = corrected_comparison[
    corrected_comparison["control"] == "unadjusted"
].iloc[0]
bulk_corrected = corrected_comparison[
    corrected_comparison["control"] == "bulk days excluded"
].iloc[0]
channel_comparison = pd.DataFrame(
    [
        {
            "measurement": "GCN observation -> circular publication",
            "median_days": gcn_channel_median_hours / 24.0,
            "p90_days": gcn_channel_p90_hours / 24.0,
            "relative_to_gcn_median": 1.0,
        },
        {
            "measurement": "SkyPortal other-origin, unadjusted",
            "median_days": unadjusted_corrected["new_other_days"],
            "p90_days": np.nan,
            "relative_to_gcn_median": (
                unadjusted_corrected["new_other_days"]
                / (gcn_channel_median_hours / 24.0)
            ),
        },
        {
            "measurement": "SkyPortal other-origin, bulk days excluded",
            "median_days": bulk_corrected["new_other_days"],
            "p90_days": np.nan,
            "relative_to_gcn_median": (
                bulk_corrected["new_other_days"]
                / (gcn_channel_median_hours / 24.0)
            ),
        },
        {
            "measurement": "SkyPortal transcription after GCN publication",
            "median_days": skyportal_transcription_median_days,
            "p90_days": decomposed_gcn[
                "publication_to_skyportal_hours"
            ].quantile(0.90)
            / 24.0,
            "relative_to_gcn_median": (
                skyportal_transcription_median_days
                / (gcn_channel_median_hours / 24.0)
            ),
        },
    ]
)
print("Channel-speed comparison:")
print(channel_comparison.round(3).to_string(index=False))
print()
print("Unresolved evidence limits:")
print("- 10/958 matched rows have negative observation-to-publication lag.")
print("- 4 GCN-origin rows reference circular files absent from the archive.")
print("- altdata and origin disagree for 302 rows across the two signals.")
print("- raw files establish timing patterns, not the operational cause of each batch.")

Channel-speed comparison:
                                  measurement  median_days  p90_days  relative_to_gcn_median
      GCN observation -> circular publication        0.447     2.428                   1.000
           SkyPortal other-origin, unadjusted       45.305       NaN                 101.319
   SkyPortal other-origin, bulk days excluded        5.808       NaN                  12.989
SkyPortal transcription after GCN publication        3.196   227.030                   7.147

Unresolved evidence limits:
- 10/958 matched rows have negative observation-to-publication lag.
- 4 GCN-origin rows reference circular files absent from the archive.
- altdata and origin disagree for 302 rows across the two signals.
- raw files establish timing patterns, not the operational cause of each batch.


**FINDING AND DECISION.** The two-speeds finding is supported for this
frozen sample when each channel is measured at first availability. The matched
GCN channel publishes at a median of 10.732 hours (p90 2.428 days). Corrected
other-origin SkyPortal rows appear after a median of 45.305 days unadjusted and
5.808 days even when the ten bulk dates are excluded: approximately 101.3 and
13.0 times the GCN median, respectively. Separately, copying published GCN
measurements into SkyPortal takes a median of 3.196 days.

The bulk dates are not one isolated migration artifact: they recur across 1,351
days and are source-concentrated. They are evidence of the retrospective
SkyPortal ingestion mechanism represented in this capture, although raw files
do not identify the operational process behind each batch. Support is therefore
partial rather than universal: 10 negative publication lags, four unavailable
circular references, provenance disagreement, and the matched-row subset limit
generalization.

## Decisions summary

The table records the revised evidence-backed decisions after separating GCN
publication from SkyPortal transcription.

In [12]:
decisions = pd.DataFrame(
    [
        (
            "Measure GCN availability at circular publication",
            "Median observation-to-publication lag is 10.732 h",
            5,
        ),
        (
            "Separate SkyPortal transcription from GCN publication",
            "Median publication-to-SkyPortal lag is 3.196 d",
            5,
        ),
        (
            "Treat bulk dates as recurring observed ingestion behavior",
            "Ten dates span 1,351 d and are source-concentrated",
            6,
        ),
        (
            "Use origin or GCN altdata as the reconciled signal",
            "Union contains 1,269 GCN-signaled rows",
            7,
        ),
        (
            "Support two speeds for the frozen sample with caveats",
            "GCN median is 0.447 d versus 5.808 d after bulk exclusion",
            8,
        ),
        (
            "Retain unresolved reference and timing failures",
            "10 negative publication lags and 4 unmatched rows remain",
            8,
        ),
    ],
    columns=["decision", "evidence", "section"],
)
print(decisions.to_string(index=False))

                                                 decision                                                  evidence  section
         Measure GCN availability at circular publication         Median observation-to-publication lag is 10.732 h        5
    Separate SkyPortal transcription from GCN publication            Median publication-to-SkyPortal lag is 3.196 d        5
Treat bulk dates as recurring observed ingestion behavior        Ten dates span 1,351 d and are source-concentrated        6
       Use origin or GCN altdata as the reconciled signal                    Union contains 1,269 GCN-signaled rows        7
    Support two speeds for the frozen sample with caveats GCN median is 0.447 d versus 5.808 d after bulk exclusion        8
          Retain unresolved reference and timing failures  10 negative publication lags and 4 unmatched rows remain        8


## 9. Is the circular matching trustworthy?

**QUESTION.** Does the regex-based `altdata` matcher identify the circular that
actually reports each photometric measurement, and how sensitive is the GCN
publication-lag estimate to implausible matches?

In [13]:
CIRCULAR_EXTRACTION_SOURCE = r"""CIRCULAR_REFERENCE_PATTERNS = [
    re.compile(r"https?://gcn\.nasa\.gov/circulars/(\d+)", re.IGNORECASE),
    re.compile(
        r"\bGCN(?:\s+CIRCULAR)?\s*[#:]?\s*(\d{4,6})\b",
        re.IGNORECASE,
    ),
]


def circular_references(altdata):
    if not isinstance(altdata, dict):
        return []
    text = json.dumps(altdata, ensure_ascii=False)
    references = []
    for pattern in CIRCULAR_REFERENCE_PATTERNS:
        references.extend(int(value) for value in pattern.findall(text))
    return sorted(set(references))
"""
print("Exact circular-identifier extraction code:")
print(CIRCULAR_EXTRACTION_SOURCE)

matched_keys = set(
    zip(decomposed_gcn["source_id"], decomposed_gcn["photometry_id"])
)
gcn_text_rows = photometry[
    photometry["has_gcn_altdata_reference"]
].copy()
gcn_text_rows["extracted_references"] = gcn_text_rows["altdata"].map(
    circular_references
)
gcn_text_rows["matched_current_rule"] = [
    (source_id, photometry_id) in matched_keys
    for source_id, photometry_id in zip(
        gcn_text_rows["source_id"], gcn_text_rows["id"]
    )
]


def nonmatch_reason(row):
    if row["provenance"] != "GCN-origin":
        return "origin_not_gcn"
    if len(row["extracted_references"]) != 1:
        return "no_unambiguous_circular_id"
    circular_path = GCN_ARCHIVE / f"{row['extracted_references'][0]}.json"
    if not circular_path.exists():
        return "circular_file_missing"
    return "matched"


gcn_text_rows["match_reason"] = gcn_text_rows.apply(
    nonmatch_reason, axis=1
)
print("GCN-text matching outcomes:")
print(
    gcn_text_rows["match_reason"]
    .value_counts()
    .rename_axis("outcome")
    .reset_index(name="rows")
    .to_string(index=False)
)

matched_altdata_examples = (
    gcn_text_rows[gcn_text_rows["matched_current_rule"]]
    .sort_values(["source_id", "id"], kind="mergesort")
    .head(5)
)
unmatched_rows = gcn_text_rows[
    ~gcn_text_rows["matched_current_rule"]
].copy()
unmatched_parts = []
for reason, count in [
    ("origin_not_gcn", 2),
    ("no_unambiguous_circular_id", 2),
    ("circular_file_missing", 1),
]:
    unmatched_parts.append(
        unmatched_rows[unmatched_rows["match_reason"] == reason]
        .sort_values(["source_id", "id"], kind="mergesort")
        .head(count)
    )
unmatched_altdata_examples = pd.concat(
    unmatched_parts, ignore_index=False
)

print()
print("Five matched altdata values:")
for _, row in matched_altdata_examples.iterrows():
    print(
        f"source_id={row['source_id']} row_id={row['id']} "
        f"origin={json.dumps(row['origin'])} "
        f"references={row['extracted_references']}"
    )
    print(json.dumps(row["altdata"], indent=2, ensure_ascii=False))

print()
print("Five GCN-text altdata values that did not match:")
for _, row in unmatched_altdata_examples.iterrows():
    print(
        f"source_id={row['source_id']} row_id={row['id']} "
        f"origin={json.dumps(row['origin'])} "
        f"reason={row['match_reason']} "
        f"references={row['extracted_references']}"
    )
    print(json.dumps(row["altdata"], indent=2, ensure_ascii=False))

Exact circular-identifier extraction code:
CIRCULAR_REFERENCE_PATTERNS = [
    re.compile(r"https?://gcn\.nasa\.gov/circulars/(\d+)", re.IGNORECASE),
    re.compile(
        r"\bGCN(?:\s+CIRCULAR)?\s*[#:]?\s*(\d{4,6})\b",
        re.IGNORECASE,
    ),
]


def circular_references(altdata):
    if not isinstance(altdata, dict):
        return []
    text = json.dumps(altdata, ensure_ascii=False)
    references = []
    for pattern in CIRCULAR_REFERENCE_PATTERNS:
        references.extend(int(value) for value in pattern.findall(text))
    return sorted(set(references))



GCN-text matching outcomes:
                   outcome  rows
                   matched   958
            origin_not_gcn   272
no_unambiguous_circular_id     5
     circular_file_missing     4

Five matched altdata values:
source_id=2025gcz row_id=60080 origin="GCN" references=[39889]
{
  "note": "GCN39889 SAO, the time corresponds to the first ob"
}
source_id=2025gcz row_id=60081 origin="GCN" references=[39890]
{
  "note": "GCN39890, SVOM/VT, arbitrary error and upper limit, note that it is not toally Johson R and the mag are reported in AB"
}
source_id=2025gcz row_id=60082 origin="GCN" references=[39890]
{
  "note": "GCN39890, SVOM/VT, arbitrary error and upper limit, note that it is not toally Johson R and the mag are reported in AB"
}
source_id=2025gcz row_id=60102 origin="GCN" references=[39894]
{
  "note": "GCN 39894 1.6m Mephisto optical observations, arbitrary upper limit",
  "exposure": "3x50"
}
source_id=2025gcz row_id=60103 origin="GCN" references=[39894]
{
  "note": "GCN 39

**MATCHING RULE.** The code serializes dictionary `altdata`, extracts four-to-six
digit identifiers from a GCN circular URL or a case-insensitive `GCN [CIRCULAR]`
marker, deduplicates them, and the current match additionally requires normalized
GCN origin, exactly one identifier, and an existing circular JSON file.

Of 1,239 GCN-text rows, 958 match; the other 281 comprise 272 rows excluded by
origin, five without an unambiguous numeric identifier, and four whose circular
file is unavailable.

In [14]:
decomposed_gcn["observation_year"] = decomposed_gcn[
    "observation_time_utc"
].dt.year
decomposed_gcn["publication_year"] = decomposed_gcn[
    "circular_published_at"
].dt.year
decomposed_gcn["publication_year_difference"] = (
    decomposed_gcn["publication_year"]
    - decomposed_gcn["observation_year"]
).abs()
decomposed_gcn["year_screen"] = np.select(
    [
        decomposed_gcn["publication_year_difference"] == 0,
        decomposed_gcn["publication_year_difference"] == 1,
    ],
    ["0 years", "1 year"],
    default=">1 year",
)
year_screen_counts = pd.DataFrame(
    {
        "year_difference": ["0 years", "1 year", ">1 year"],
        "rows": [
            int((decomposed_gcn["publication_year_difference"] == 0).sum()),
            int((decomposed_gcn["publication_year_difference"] == 1).sum()),
            int((decomposed_gcn["publication_year_difference"] > 1).sum()),
        ],
    }
)
print("Publication-year plausibility screen:")
print(year_screen_counts.to_string(index=False))

implausible_year_rows = decomposed_gcn[
    decomposed_gcn["publication_year_difference"] > 1
].sort_values(
    ["publication_year_difference", "source_id", "photometry_id"],
    ascending=[False, True, True],
    kind="mergesort",
)
print()
print("All rows differing by more than one publication year:")
for _, row in implausible_year_rows.iterrows():
    print(
        json.dumps(
            {
                "source_id": row["source_id"],
                "photometry_id": int(row["photometry_id"]),
                "mjd": float(row["mjd"]),
                "observation_date": row[
                    "observation_time_utc"
                ].isoformat(),
                "circular_id": int(row["circular_id"]),
                "publication_date": row[
                    "circular_published_at"
                ].isoformat(),
                "lag_A_hours": float(
                    row["observation_to_publication_hours"]
                ),
            },
            indent=2,
            ensure_ascii=False,
        )
    )

Publication-year plausibility screen:
year_difference  rows
        0 years   952
         1 year     4
        >1 year     2



All rows differing by more than one publication year:
{
  "source_id": "GRB-260101_005630",
  "photometry_id": 83197,
  "mjd": 61041.08001157407,
  "observation_date": "2026-01-01T01:55:12.999999751+00:00",
  "circular_id": 4386,
  "publication_date": "2005-12-22T12:11:37+00:00",
  "lag_A_hours": -175549.7266666666
}
{
  "source_id": "GRB-260101_005630",
  "photometry_id": 83198,
  "mjd": 61041.092824074076,
  "observation_date": "2026-01-01T02:13:40.000000157+00:00",
  "circular_id": 4287,
  "publication_date": "2005-11-17T19:32:08+00:00",
  "lag_A_hours": -176382.69222222225
}


**PLAUSIBILITY FINDING.** Publication and observation years agree for 952
rows, differ by one year for four, and differ by more than one year for two.
The two failures connect 2026 observations to circular ids published in 2005 and
are also among the negative lag-A rows.

In [15]:
manual_quantiles = [
    ("p10", 0.10),
    ("p25", 0.25),
    ("p50", 0.50),
    ("p75", 0.75),
    ("p90", 0.90),
    ("max", None),
]
print("Manual-verification evidence at fixed lag-A quantiles:")
for quantile_name, quantile_value in manual_quantiles:
    if quantile_value is None:
        selected = decomposed_gcn.sort_values(
            ["observation_to_publication_hours", "source_id", "photometry_id"],
            ascending=[False, True, True],
            kind="mergesort",
        ).iloc[0]
        target = selected["observation_to_publication_hours"]
    else:
        target = decomposed_gcn[
            "observation_to_publication_hours"
        ].quantile(quantile_value)
        selected = (
            decomposed_gcn.assign(
                distance=(
                    decomposed_gcn["observation_to_publication_hours"]
                    - target
                ).abs()
            )
            .sort_values(
                [
                    "distance",
                    "observation_to_publication_hours",
                    "source_id",
                    "photometry_id",
                ],
                kind="mergesort",
            )
            .iloc[0]
        )
    circular_path = GCN_ARCHIVE / f"{int(selected['circular_id'])}.json"
    circular = json.loads(circular_path.read_text(encoding="utf-8"))
    evidence_record = {
        "quantile": quantile_name,
        "target_lag_A_hours": float(target),
        "source_id": selected["source_id"],
        "photometry_id": int(selected["photometry_id"]),
        "mjd": float(selected["mjd"]),
        "observation_time_utc": selected[
            "observation_time_utc"
        ].isoformat(),
        "altdata": selected["raw_record"].get("altdata"),
        "matched_circular_id": int(selected["circular_id"]),
        "circular_subject": circular.get("subject"),
        "circular_publication_date": selected[
            "circular_published_at"
        ].isoformat(),
        "circular_body_first_400_characters": circular.get("body", "")[:400],
        "lag_A_hours": float(
            selected["observation_to_publication_hours"]
        ),
    }
    print(json.dumps(evidence_record, indent=2, ensure_ascii=False))

Manual-verification evidence at fixed lag-A quantiles:
{
  "quantile": "p10",
  "target_lag_A_hours": 1.1408246388638612,
  "source_id": "EP-260623_025405",
  "photometry_id": 96204,
  "mjd": 61214.13064,
  "observation_time_utc": "2026-06-23T03:08:07.296000259+00:00",
  "altdata": {
    "GCN": "https://gcn.nasa.gov/circulars/45021"
  },
  "matched_circular_id": 45021,
  "circular_subject": "EP260623a: Las Cumbres discovery of the optical counterpart",
  "circular_publication_date": "2026-06-23T04:16:44.306000+00:00",
  "circular_body_first_400_characters": "Wenxiong Li, Runduo Liang (NAOC), Iair Arcavi, Ido Keinan (TAU), David Sand (U of Arizona)\n\nWe observed the position of EP260623a (Wang et al. GCN 45020) with a Las Cumbres 1m telescope at Teide Observatory, Tenerife, 8 mins after the Einstein Probe WXT trigger. We took 2x300s exposures in the broad optical w-band.\nWe find an uncataloged source at RA=328.2698, Dec=12.7939 within the EP-FXT error c",
  "lag_A_hours": 1.1436138888

The six records above are printed as audit evidence only. Their subjects,
publication dates, and body excerpts are left for human assessment without an
automated plausibility judgment.

In [16]:
sensitivity_sets = [
    ("all matched rows", decomposed_gcn),
    (
        "exclude negative lag A",
        decomposed_gcn[
            decomposed_gcn["observation_to_publication_hours"] >= 0
        ],
    ),
    (
        "exclude negative lag A and >1-year failures",
        decomposed_gcn[
            (decomposed_gcn["observation_to_publication_hours"] >= 0)
            & (decomposed_gcn["publication_year_difference"] <= 1)
        ],
    ),
]
sensitivity_rows = []
for screen, selected in sensitivity_sets:
    median_hours = selected[
        "observation_to_publication_hours"
    ].median()
    p90_hours = selected[
        "observation_to_publication_hours"
    ].quantile(0.90)
    sensitivity_rows.append(
        {
            "screen": screen,
            "n": len(selected),
            "median_hours": median_hours,
            "median_days": median_hours / 24.0,
            "p90_hours": p90_hours,
            "p90_days": p90_hours / 24.0,
        }
    )
matching_sensitivity = pd.DataFrame(sensitivity_rows)
print("Lag-A sensitivity to plausibility exclusions:")
print(matching_sensitivity.round(3).to_string(index=False))

Lag-A sensitivity to plausibility exclusions:
                                     screen   n  median_hours  median_days  p90_hours  p90_days
                           all matched rows 958        10.732        0.447     58.269     2.428
                     exclude negative lag A 948        11.047        0.460     59.274     2.470
exclude negative lag A and >1-year failures 948        11.047        0.460     59.274     2.470


**SENSITIVITY FINDING.** Matching is not fully trustworthy at row level:
10/958 lag-A values are negative and two fail the greater-than-one-year screen.
The distributional estimate is stable, however. Excluding negatives moves the
median from 10.732 to 11.047 hours and p90 from 58.269 to 59.274 hours; the year
screen removes no additional row because both greater-than-one-year failures are
already negative.

## 10. Are the bulk dates retrospective or campaign surges?

**QUESTION.** Do the ten high-volume creation dates primarily contain same-day
campaign photometry or older data loaded retrospectively?

In [17]:
bulk_audit_rows = []
for created_day in top_upload_days["created_day"]:
    day_rows = photometry[photometry["created_day"] == created_day].copy()
    age_days = day_rows["lag_hours"] / 24.0
    source_counts = (
        day_rows["source_id"]
        .value_counts()
        .rename_axis("source_id")
        .reset_index(name="rows")
        .sort_values(
            ["rows", "source_id"],
            ascending=[False, True],
            kind="mergesort",
        )
    )
    median_age_days = age_days.median()
    if median_age_days < 2:
        bulk_class = "SAME-DAY CAMPAIGN"
    elif median_age_days > 30:
        bulk_class = "RETROSPECTIVE LOAD"
    else:
        bulk_class = "MIXED"
    origin_breakdown = {
        str(origin): int(count)
        for origin, count in day_rows["origin"]
        .value_counts()
        .sort_index()
        .items()
    }
    bulk_audit_rows.append(
        {
            "date": created_day,
            "rows": len(day_rows),
            "distinct_source_ids": day_rows["source_id"].nunique(),
            "top_source_id": source_counts.iloc[0]["source_id"],
            "top_source_rows": int(source_counts.iloc[0]["rows"]),
            "age_min_days": age_days.min(),
            "age_median_days": median_age_days,
            "age_max_days": age_days.max(),
            "origin_breakdown": json.dumps(
                origin_breakdown, ensure_ascii=False, sort_keys=True
            ),
            "classification": bulk_class,
        }
    )
bulk_date_audit = pd.DataFrame(bulk_audit_rows)
print("Bulk-date age and source audit:")
print(
    bulk_date_audit.round(
        {
            "age_min_days": 3,
            "age_median_days": 3,
            "age_max_days": 3,
        }
    ).to_string(index=False)
)

classification_order = [
    "SAME-DAY CAMPAIGN",
    "RETROSPECTIVE LOAD",
    "MIXED",
]
classification_rows = []
for bulk_class in classification_order:
    selected_dates = bulk_date_audit[
        bulk_date_audit["classification"] == bulk_class
    ]
    classification_rows.append(
        {
            "classification": bulk_class,
            "dates": len(selected_dates),
            "rows": int(selected_dates["rows"].sum()),
        }
    )
bulk_classification_summary = pd.DataFrame(classification_rows)
print()
print("Bulk rows by median-age class:")
print(bulk_classification_summary.to_string(index=False))

retrospective_dates = bulk_date_audit.loc[
    bulk_date_audit["classification"] == "RETROSPECTIVE LOAD",
    "date",
]
corrected_other_rows = photometry[
    photometry["corrected_provenance"] == "other-origin"
]
exclude_all_bulk_median_days = corrected_other_rows[
    ~corrected_other_rows["created_day"].isin(top_upload_days["created_day"])
]["lag_hours"].median() / 24.0
exclude_retrospective_median_days = corrected_other_rows[
    ~corrected_other_rows["created_day"].isin(retrospective_dates)
]["lag_hours"].median() / 24.0
bulk_exclusion_sensitivity = pd.DataFrame(
    [
        {
            "screen": "exclude all ten bulk dates",
            "other_origin_median_days": exclude_all_bulk_median_days,
        },
        {
            "screen": "exclude retrospective-load dates only",
            "other_origin_median_days": exclude_retrospective_median_days,
        },
    ]
)
print()
print("Other-origin median under bulk-date exclusions:")
print(bulk_exclusion_sensitivity.round(3).to_string(index=False))

Bulk-date age and source audit:
      date  rows  distinct_source_ids         top_source_id  top_source_rows  age_min_days  age_median_days  age_max_days                                    origin_breakdown     classification
2024-04-26   742                    1             EP240426a              742         0.143          121.383       395.550                           {"fp": 741, "stdview": 1} RETROSPECTIVE LOAD
2024-07-16   740                    2 IceCubeCascade240714A              382         1.506           89.560       173.157                                         {"fp": 740} RETROSPECTIVE LOAD
2024-06-27   642                    4          ZTF23abvvlla              493         0.494           99.461       395.591                {"GRANDMA": 1, "None": 1, "fp": 640} RETROSPECTIVE LOAD
2022-11-10   469                    1             GRB221110              469         0.627           84.313       370.234                                       {"None": 469} RETROSPECTIVE LOAD
202


Other-origin median under bulk-date exclusions:
                               screen  other_origin_median_days
           exclude all ten bulk dates                     5.808
exclude retrospective-load dates only                     6.447


**FINDING.** Nine dates containing 3,507 rows are retrospective loads by
median age, one date containing 157 rows is mixed, and none meets the same-day
campaign threshold. Excluding only retrospective dates gives a corrected
other-origin median of 6.447 days, compared with 5.808 days when all ten dates
are excluded.

The bulk dates are therefore heterogeneous at the threshold boundary but
overwhelmingly retrospective by row count. On 2024-04-26, all 742 rows belong to
`EP240426a`, and the ingestion date matches the date encoded in that event name;
nevertheless, the rows span observation ages from 0.143 to 395.550 days with a
121.383-day median. This table supports only those observed facts and does not
identify the operational cause.